In [2]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from google.colab import files

uploaded = files.upload()  # Select english_hinglish_sample_dataset.csv
file_name = next(name for name in uploaded if name.lower().endswith(".csv"))

df = pd.read_csv(file_name)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Saving english_hinglish_sample_dataset.csv to english_hinglish_sample_dataset (1).csv
Dataset shape: (152, 2)
Columns: ['english', 'hinglish']


,english,hinglish
0,Hello,Namaste
1,Hi,Hi
2,Good morning,Good morning
3,Good afternoon,Good afternoon
4,Good evening,Good evening


In [3]:
SOURCE_COL = "english"
TARGET_COL = "hinglish"

if SOURCE_COL not in df.columns or TARGET_COL not in df.columns:
    raise ValueError(
        f"CSV must contain columns '{SOURCE_COL}' and '{TARGET_COL}'. "
        f"Found: {df.columns.tolist()}"
    )

data = df[[SOURCE_COL, TARGET_COL]].dropna().copy()

def clean_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9?!.', ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data[SOURCE_COL] = data[SOURCE_COL].apply(clean_text)
data[TARGET_COL] = data[TARGET_COL].apply(clean_text)
data = data[(data[SOURCE_COL] != "") & (data[TARGET_COL] != "")]
data = data.drop_duplicates().reset_index(drop=True)

# Add decoder boundary tokens
data[TARGET_COL] = "startseq " + data[TARGET_COL] + " endseq"

print("Sentence pairs after cleaning:", len(data))
display(data.head())

Sentence pairs after cleaning: 152


,english,hinglish
0,hello,startseq namaste endseq
1,hi,startseq hi endseq
2,good morning,startseq good morning endseq
3,good afternoon,startseq good afternoon endseq
4,good evening,startseq good evening endseq


In [4]:
from tensorflow.keras.layers import TextVectorization

MAX_VOCAB = 5000
MAX_LEN = 20

source_vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
    standardize=None
)
target_vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
    standardize=None
)

source_vectorizer.adapt(data[SOURCE_COL].values)
target_vectorizer.adapt(data[TARGET_COL].values)

X = source_vectorizer(data[SOURCE_COL].values).numpy()
Y = target_vectorizer(data[TARGET_COL].values).numpy()

source_vocab = source_vectorizer.get_vocabulary()
target_vocab = target_vectorizer.get_vocabulary()
source_vocab_size = len(source_vocab)
target_vocab_size = len(target_vocab)

target_word_to_id = {word: i for i, word in enumerate(target_vocab)}
target_id_to_word = {i: word for i, word in enumerate(target_vocab)}
start_id = target_word_to_id.get("startseq")
end_id = target_word_to_id.get("endseq")

print("Source vocabulary size:", source_vocab_size)
print("Target vocabulary size:", target_vocab_size)
print("English sequence shape:", X.shape)
print("Target sequence shape:", Y.shape)
print("Start token ID:", start_id, "| End token ID:", end_id)

Source vocabulary size: 237
Target vocabulary size: 263
English sequence shape: (152, 20)
Target sequence shape: (152, 20)
Start token ID: 2 | End token ID: 3


# New Section

In [5]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(data))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42
)

X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]
train_df = data.iloc[train_idx].reset_index(drop=True)
test_df = data.iloc[test_idx].reset_index(drop=True)

# Decoder inputs contain startseq and previous target words.
decoder_train_input = Y_train[:, :-1]
decoder_train_target = Y_train[:, 1:]
decoder_test_input = Y_test[:, :-1]
decoder_test_target = Y_test[:, 1:]

train_weights = (decoder_train_target != 0).astype("float32")
test_weights = (decoder_test_target != 0).astype("float32")

print("Training pairs:", len(X_train))
print("Testing pairs:", len(X_test))

Training pairs: 121
Testing pairs: 31


# New Section

In [6]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(data))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42
)

X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]
train_df = data.iloc[train_idx].reset_index(drop=True)
test_df = data.iloc[test_idx].reset_index(drop=True)

# Decoder inputs contain startseq and previous target words.
decoder_train_input = Y_train[:, :-1]
decoder_train_target = Y_train[:, 1:]
decoder_test_input = Y_test[:, :-1]
decoder_test_target = Y_test[:, 1:]

train_weights = (decoder_train_target != 0).astype("float32")
test_weights = (decoder_test_target != 0).astype("float32")

print("Training pairs:", len(X_train))
print("Testing pairs:", len(X_test))

Training pairs: 121
Testing pairs: 31


# New Section

In [7]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

EMBED_DIM = 128
LATENT_DIM = 128

# Encoder
encoder_inputs = Input(shape=(MAX_LEN,), name="encoder_inputs")
encoder_embedding = Embedding(
    source_vocab_size, EMBED_DIM, mask_zero=True, name="encoder_embedding"
)
encoder_embedded = encoder_embedding(encoder_inputs)
encoder_lstm = LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(encoder_embedded)
encoder_states = [state_h, state_c]

# Decoder for training
decoder_inputs = Input(shape=(MAX_LEN - 1,), name="decoder_inputs")
decoder_embedding = Embedding(
    target_vocab_size, EMBED_DIM, mask_zero=True, name="decoder_embedding"
)
decoder_embedded = decoder_embedding(decoder_inputs)
decoder_lstm = LSTM(
    LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm"
)
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedded, initial_state=encoder_states
)
decoder_dense = Dense(target_vocab_size, activation="softmax", name="word_output")
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    weighted_metrics=["sparse_categorical_accuracy"]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 19)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 20, 128)   │     30,336 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 20)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 19, 128)   │     33,664 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │    131,584 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 19, 128), │    131,584 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_output (Dense) │ (None, 19, 263)   │     33,927 │ decoder_lstm[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 361,095 (1.38 MB)

 Trainable params: 361,095 (1.38 MB)

 Non-trainable params: 0 (0.00 B)

# New Section

In [8]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)

history = model.fit(
    [X_train, decoder_train_input],
    decoder_train_target,
    sample_weight=train_weights,
    validation_data=(
        [X_test, decoder_test_input],
        decoder_test_target,
        test_weights
    ),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 145ms/step - loss: 4.5863 - sparse_categorical_accuracy: 0.1620 - val_loss: 4.5744 - val_sparse_categorical_accuracy: 0.2067
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 4.4868 - sparse_categorical_accuracy: 0.2108 - val_loss: 4.3671 - val_sparse_categorical_accuracy: 0.2067
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 3.9786 - sparse_categorical_accuracy: 0.2108 - val_loss: 3.8188 - val_sparse_categorical_accuracy: 0.2067
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - loss: 3.6529 - sparse_categorical_accuracy: 0.2108 - val_loss: 3.8101 - val_sparse_categorical_accuracy: 0.2067
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - loss: 3.4868 - sparse_categorical_accuracy: 0.2108 - val_loss: 3.8550 - val_sparse_categorical_accuracy: 0.2133
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 3.4191 - sparse_categorical_accuracy: 0.2387 - val_loss: 3.9042 - val_sparse_categorical_accuracy: 0.2333
Epoch 7/100
8/8

# New Section

In [9]:
results = model.evaluate(
    [X_test, decoder_test_input],
    decoder_test_target,
    sample_weight=test_weights,
    verbose=0,
    return_dict=True
)
print("Test loss:", round(float(results["loss"]), 4))
for key, value in results.items():
    if key != "loss":
        print(key + ":", round(float(value), 4))

Test loss: 3.5966
sparse_categorical_accuracy: 0.3133


# New Section

In [10]:
# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# One-step decoder inference model reusing trained layers
decoder_state_h = Input(shape=(LATENT_DIM,), name="decoder_state_h")
decoder_state_c = Input(shape=(LATENT_DIM,), name="decoder_state_c")
decoder_token_input = Input(shape=(1,), name="decoder_token_input")

step_embedded = decoder_embedding(decoder_token_input)
step_output, next_h, next_c = decoder_lstm(
    step_embedded, initial_state=[decoder_state_h, decoder_state_c]
)
step_output = decoder_dense(step_output)

decoder_model = Model(
    [decoder_token_input, decoder_state_h, decoder_state_c],
    [step_output, next_h, next_c]
)

def translate(sentence, max_steps=MAX_LEN):
    sentence = clean_text(sentence)
    input_seq = source_vectorizer(tf.constant([sentence])).numpy()
    states = encoder_model.predict(input_seq, verbose=0)

    token_id = start_id
    words = []

    for step in range(max_steps):
        output, h, c = decoder_model.predict(
            [np.array([[token_id]]), states[0], states[1]],
            verbose=0
        )
        probs = output[0, -1, :].copy()

        # Do not allow padding or startseq to be generated as output.
        probs[0] = 0
        if start_id is not None:
            probs[start_id] = 0

        # If the model immediately predicts endseq, allow one best non-special
        # word for a visible diagnostic output rather than returning blank.
        if step == 0 and end_id is not None:
            probs[end_id] = 0

        token_id = int(np.argmax(probs))
        states = [h, c]

        if end_id is not None and token_id == end_id:
            break

        word = target_id_to_word.get(token_id, "")
        if word and word not in ("", "[UNK]", "startseq", "endseq"):
            words.append(word)

    return " ".join(words) if words else "(No word predicted; train with more paired data.)"

print("Example translation:", translate("How are you?"))

Example translation: mujhe hai


# New Section

In [11]:
!pip -q install nltk

# New Section

In [12]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

references = []
hypotheses = []

for _, row in test_df.iterrows():
    reference = row[TARGET_COL].replace("startseq ", "").replace(" endseq", "")
    prediction = translate(row[SOURCE_COL])
    references.append([reference.split()])
    hypotheses.append(prediction.split() if prediction else [])

bleu = corpus_bleu(
    references, hypotheses,
    smoothing_function=SmoothingFunction().method1
)
print("BLEU Score (0 to 1):", round(float(bleu), 4))
print("BLEU Percentage:", round(float(bleu * 100), 2), "%")
print("Note: BLEU can be low for a tiny dataset; report the measured score.")

BLEU Score (0 to 1): 0.009
BLEU Percentage: 0.9 %
Note: BLEU can be low for a tiny dataset; report the measured score.


# New Section

In [13]:
test_sentences = [
    "How are you?",
    "What is your name?",
    "I am going to college.",
    "Please help me.",
    "Today is a beautiful day."
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Hinglish prediction:", translate(sentence))
    print("-" * 40)

English: How are you?
Hinglish prediction: mujhe hai
----------------------------------------
English: What is your name?
Hinglish prediction: mujhe mujhe hai
----------------------------------------
English: I am going to college.
Hinglish prediction: main main hoon
----------------------------------------
English: Please help me.
Hinglish prediction: kripya hai
----------------------------------------
English: Today is a beautiful day.
Hinglish prediction: kripya mujhe hai
----------------------------------------
